# Maths Verification Notebook

A companion to the [Report card](../README.md) tracker for Advanced Higher Maths.

The idea is the same as running a problem through a CAS like *Derive*: instead of trusting an
answer you worked out by hand, you re-derive it symbolically here and let SymPy confirm the two
match. Use it after a past paper or homework set — paste in the question's expression and your
worked answer, run the cell, and get a clear **PASS/FAIL** instead of a guess.

**How it works**

- `verify(expression, claimed_answer)` tries to prove the two are symbolically identical
  (`simplify`, then `equals`), and falls back to checking numerically at several random points
  when a symbolic proof doesn't resolve — this catches most transcription/sign/simplification
  mistakes without needing a fully closed-form proof for messy trig or log expressions.
- **When it's a FAIL, it doesn't just say so.** It walks back through the correct derivation
  step by step (pass your own steps in via `steps=[...]`, as every worked example below does),
  then works out *how* your answer differs from the correct one — a leftover constant, a
  missing or sign-flipped term — and points you at the specific line worth re-checking, instead
  of leaving you to find the mistake blind.
- Each syllabus section below (differentiation, integration, equations, sequences & series,
  complex numbers, vectors) has worked examples showing the pattern, plus a blank template at
  the end for your own problems.


In [1]:
import sympy as sp
from sympy import (
    symbols, Symbol, diff, integrate, simplify, solve, solveset, Eq,
    sqrt, sin, cos, tan, exp, log, pi, I, re, im, Abs, arg, oo, S,
    Sum, summation, Rational, nsimplify, factor, expand, together, cancel, Matrix
)
import random

x, y, z, t, n, k, theta = symbols('x y z t n k theta', real=True)

def _diagnose(residual, reference_expr=None):
    """Best-effort guess at *what kind* of slip produced this residual."""
    residual = sp.nsimplify(residual) if residual.is_number else residual
    if residual == 0:
        return "No numeric difference found — re-check the comparison by hand."
    if residual.is_number:
        return (f"Your answer differs from the correct one by a constant, {residual}. "
                f"Look for a dropped '+ C', a boundary term, or an arithmetic slip in the final step.")
    if reference_expr is not None:
        terms = sp.Add.make_args(sp.expand(sp.sympify(reference_expr)))
        for term in terms:
            if term == 0:
                continue
            if sp.simplify(residual + term) == 0:
                return (f"This looks like a sign error on the term {term} in your working — "
                        f"check whether it should be negative instead of positive (or vice versa).")
            if sp.simplify(residual - term) == 0:
                return f"You seem to be missing the term {term} — check whether it got dropped while simplifying."
    factored = sp.factor(sp.together(residual))
    return (f"The gap between your answer and the correct one simplifies to {factored}. "
            f"Retrace the steps above line by line and compare against this — that's where it'll show up.")

def _report(verdict, label, your_answer, correct_answer, *, steps=None, diagnosis=None, method=None):
    tag = "PASS" if verdict is True else ("FAIL" if verdict is False else "INCONCLUSIVE")
    icon = {"PASS": "\u2713", "FAIL": "\u2717", "INCONCLUSIVE": "?"}[tag]
    print(f"{icon} {tag}" + (f" — {label}" if label else ""))
    print(f"  your answer:   {your_answer}")
    print(f"  correct form:  {correct_answer}")
    if method:
        print(f"  method:        {method}")
    if verdict is False:
        print()
        print("Let's work through the correct derivation:")
        if steps:
            for i, s in enumerate(steps, 1):
                print(f"  {i}. {s}")
        else:
            print("  (no steps recorded for this one — recompute from the definition/rule and")
            print("   compare each line of your working against the correct form above)")
        if diagnosis:
            print()
            print("Where to look for the mistake:")
            print(f"  {diagnosis}")
    elif verdict is None:
        print("  Could not confirm either way automatically — check this one by hand.")
    print()
    return verdict

def verify(expr_a, expr_b, *, label="", steps=None, domain_vars=None, samples=25, tol=1e-9):
    """Check whether expr_a (the correct, derived expression) equals expr_b (your answer).

    Tries an exact symbolic proof first (difference simplifies to 0, or .equals()). If SymPy
    can't decide symbolically (common with trig/log/piecewise forms), falls back to substituting
    random numeric values for every free symbol and checking the two sides agree everywhere
    they're both defined.

    On a FAIL, prints the full step-by-step derivation (from `steps`, if you supply it) and a
    diagnosis of how your answer differs from the correct one, instead of just flagging it wrong.
    """
    expr_a = sp.sympify(expr_a)
    expr_b = sp.sympify(expr_b)
    diff_expr = sp.together(expr_a - expr_b)

    verdict = None
    method = None

    simplified = sp.simplify(diff_expr)
    if simplified == 0:
        verdict, method = True, "symbolic simplify"
    else:
        eq = expr_a.equals(expr_b)
        if eq is True:
            verdict, method = True, "symbolic .equals()"
        elif eq is False:
            verdict, method = False, "symbolic .equals()"

    if verdict is None:
        free = sorted((expr_a.free_symbols | expr_b.free_symbols), key=str)
        free = domain_vars if domain_vars else free
        checks, mismatches = 0, 0
        for _ in range(samples):
            subs = {s: Rational(random.randint(-500, 500), random.randint(1, 37)) for s in free}
            try:
                va = complex(expr_a.evalf(subs=subs))
                vb = complex(expr_b.evalf(subs=subs))
            except (TypeError, ValueError):
                continue
            if any(map(lambda v: v != v, (va.real, va.imag, vb.real, vb.imag))):
                continue
            checks += 1
            if abs(va - vb) > tol * max(1, abs(va), abs(vb)):
                mismatches += 1
        if checks == 0:
            verdict, method = None, "could not evaluate numerically"
        else:
            verdict, method = (mismatches == 0), f"numeric check ({checks} random points)"

    diagnosis = None
    if verdict is False:
        residual = sp.simplify(expr_b - expr_a)
        diagnosis = _diagnose(residual, expr_a)

    correct_display = sp.radsimp(sp.simplify(expr_a))
    return _report(verdict, label, expr_b, correct_display, steps=steps, diagnosis=diagnosis, method=method)


## Differentiation

Differentiate the expression yourself, then check your answer against SymPy's derivative.

In [2]:
# Example: differentiate f(x) = x^3 * sin(2x)
f = x**3 * sin(2*x)
my_answer = 3*x**2*sin(2*x) + 2*x**3*cos(2*x)

u, v = x**3, sin(2*x)
du, dv = diff(u, x), diff(v, x)
steps = [
    f"Product rule: write f(x) = u*v with u = {u}, v = {v}.",
    f"Differentiate each factor: u' = {du} (power rule), v' = {dv} (chain rule: d/dx sin(2x) = 2cos(2x)).",
    f"Apply the product rule: f'(x) = u'v + uv' = ({du})({v}) + ({u})({dv}).",
    f"Simplify: f'(x) = {sp.simplify(du*v + u*dv)}.",
]

derived = diff(f, x)
verify(derived, my_answer, label="d/dx[x^3 sin(2x)]", steps=steps)


✓ PASS — d/dx[x^3 sin(2x)]
  your answer:   2*x**3*cos(2*x) + 3*x**2*sin(2*x)
  correct form:  x**2*(2*x*cos(2*x) + 3*sin(2*x))
  method:        symbolic simplify



True

In [3]:
# Example: a wrong answer on purpose, to show what a FAIL — and the coaching — looks like
f2 = (x**2 + 1) / (x - 1)
my_wrong_answer = (2*x*(x - 1) - (x**2 + 1)) / (x - 1)**2 + 1  # deliberate slip: +1 shouldn't be there

u2, v2 = x**2 + 1, x - 1
du2, dv2 = diff(u2, x), diff(v2, x)
numerator = sp.expand(du2*v2 - u2*dv2)
steps = [
    f"Quotient rule: for f(x) = u/v with u = {u2}, v = {v2}: f'(x) = (u'v - uv') / v^2.",
    f"u' = {du2}, v' = {dv2}.",
    f"Numerator: u'v - uv' = ({du2})({v2}) - ({u2})({dv2}) = {sp.expand(du2*v2)} - {sp.expand(u2*dv2)} = {numerator}.",
    f"So f'(x) = ({numerator}) / ({v2})**2 = {sp.simplify((du2*v2 - u2*dv2)/v2**2)}.",
]

derived2 = diff(f2, x)
verify(derived2, my_wrong_answer, label="d/dx[(x^2+1)/(x-1)] (has a deliberate mistake)", steps=steps)


✗ FAIL — d/dx[(x^2+1)/(x-1)] (has a deliberate mistake)
  your answer:   1 + (-x**2 + 2*x*(x - 1) - 1)/(x - 1)**2
  correct form:  (-x**2 + 2*x*(x - 1) - 1)/(x - 1)**2
  method:        symbolic .equals()

Let's work through the correct derivation:
  1. Quotient rule: for f(x) = u/v with u = x**2 + 1, v = x - 1: f'(x) = (u'v - uv') / v^2.
  2. u' = 2*x, v' = 1.
  3. Numerator: u'v - uv' = (2*x)(x - 1) - (x**2 + 1)(1) = 2*x**2 - 2*x - x**2 + 1 = x**2 - 2*x - 1.
  4. So f'(x) = (x**2 - 2*x - 1) / (x - 1)**2 = (-x**2 + 2*x*(x - 1) - 1)/(x - 1)**2.

Where to look for the mistake:
  Your answer differs from the correct one by a constant, 1. Look for a dropped '+ C', a boundary term, or an arithmetic slip in the final step.



False

## Integration

Same pattern — integrate by hand, check against SymPy's antiderivative. (Integration answers differ by a constant, so compare derivatives to sidestep the `+ C` question, or just check `equals()` up to a constant as shown below.)

In [4]:
# Example: integrate x * exp(x^2)
g = x * exp(x**2)
my_answer = sp.Rational(1, 2) * exp(x**2)  # + C, omitted

steps = [
    "Spot the substitution: let w = x^2, so dw = 2x dx, i.e. x dx = dw/2.",
    "Rewrite the integral: ∫ x e^(x^2) dx = ∫ e^w · (dw/2) = (1/2) ∫ e^w dw.",
    "Integrate: (1/2) e^w + C.",
    "Substitute back w = x^2: (1/2) e^(x^2) + C.",
]

derived_antideriv = integrate(g, x)
# Compare derivatives of both sides instead of the antiderivatives themselves,
# so an arbitrary "+ C" doesn't cause a false FAIL.
verify(diff(derived_antideriv, x), diff(my_answer, x), label="∫ x e^(x^2) dx", steps=steps)


✓ PASS — ∫ x e^(x^2) dx
  your answer:   x*exp(x**2)
  correct form:  x*exp(x**2)
  method:        symbolic simplify



True

In [5]:
# Definite integral — no +C ambiguity, so just check the value directly
h = sin(x)**2
a_lo, a_hi = 0, pi
my_answer_def = pi / 2

steps = [
    "Use the power-reduction identity: sin^2(x) = (1 - cos(2x)) / 2.",
    "∫ sin^2(x) dx = ∫ (1 - cos(2x))/2 dx = x/2 - sin(2x)/4 + C.",
    "Evaluate from 0 to π: [π/2 - sin(2π)/4] - [0 - sin(0)/4] = π/2 - 0 - 0 = π/2.",
]

derived_def = integrate(h, (x, a_lo, a_hi))
verify(derived_def, my_answer_def, label="∫₀^π sin²(x) dx", steps=steps)


✓ PASS — ∫₀^π sin²(x) dx
  your answer:   pi/2
  correct form:  pi/2
  method:        symbolic simplify



True

## Equations & algebra

Solve the equation yourself, then compare your root set to SymPy's.

In [6]:
# Example: solve x^3 - 6x^2 + 11x - 6 = 0
equation = x**3 - 6*x**2 + 11*x - 6
my_roots = {1, 2, 3}

steps = [
    "Try small integer factors of the constant term (-6): x=1 gives 1-6+11-6=0, so (x-1) is a factor.",
    "Divide through: x^3 - 6x^2 + 11x - 6 = (x - 1)(x^2 - 5x + 6).",
    "Factor the quadratic: x^2 - 5x + 6 = (x - 2)(x - 3).",
    "So the equation factors as (x-1)(x-2)(x-3) = 0, giving roots x = 1, 2, 3.",
]

derived_roots = set(solve(Eq(equation, 0), x))
verdict = derived_roots == my_roots
diagnosis = None
if not verdict:
    missing = derived_roots - my_roots
    extra = my_roots - derived_roots
    parts = []
    if missing:
        parts.append(f"you're missing root(s) {sorted(missing)}")
    if extra:
        parts.append(f"you have extra root(s) {sorted(extra)} that don't satisfy the equation")
    diagnosis = "; ".join(parts) + " — recheck the factoring/division step above."

_report(verdict, "roots of x^3 - 6x^2 + 11x - 6 = 0", sorted(my_roots), sorted(derived_roots),
        steps=steps, diagnosis=diagnosis)


✓ PASS — roots of x^3 - 6x^2 + 11x - 6 = 0
  your answer:   [1, 2, 3]
  correct form:  [1, 2, 3]



True

In [7]:
# Example: simplify/rationalise a surd expression and check equivalence
expr = 1 / (sqrt(5) - sqrt(3))
my_answer = (sqrt(5) + sqrt(3)) / 2

steps = [
    "Multiply top and bottom by the conjugate (√5 + √3): 1/(√5-√3) · (√5+√3)/(√5+√3).",
    "The denominator becomes a difference of squares: (√5)^2 - (√3)^2 = 5 - 3 = 2.",
    "So the expression simplifies to (√5 + √3) / 2.",
]

verify(expr, my_answer, label="rationalise 1/(√5 - √3)", steps=steps)


✓ PASS — rationalise 1/(√5 - √3)
  your answer:   sqrt(3)/2 + sqrt(5)/2
  correct form:  (sqrt(3) + sqrt(5))/2
  method:        symbolic simplify



True

## Sequences & series

Check a closed-form sum against SymPy's evaluation of the series.

In [8]:
# Example: sum of first n squares
series_expr = Sum(k**2, (k, 1, n)).doit()
my_formula = n*(n + 1)*(2*n + 1) / 6

steps = [
    "Standard result: Σ_{k=1}^n k^2 = n(n+1)(2n+1)/6, provable by induction.",
    "Base case n=1: formula gives 1·2·3/6 = 1, matching Σ = 1^2 = 1. ✓",
    "Inductive step: assume it holds for n=m; adding (m+1)^2 to both sides and factoring recovers "
    "the same formula with n=m+1, so it holds for all n ≥ 1.",
]

verify(series_expr, my_formula, label="Σ k² for k=1..n", steps=steps)


✓ PASS — Σ k² for k=1..n
  your answer:   n*(n + 1)*(2*n + 1)/6
  correct form:  n*(2*n**2 + 3*n + 1)/6
  method:        symbolic simplify



True

In [9]:
# Example: arithmetic series sum, a=3, d=2 — same pattern, swap the formula
arithmetic_sum = Sum(3 + (k - 1)*2, (k, 1, n)).doit()
my_arith_formula = n * (n + 2)

steps = [
    "Arithmetic series formula: S_n = n/2 · (2a + (n-1)d), with a=3, d=2 here.",
    "S_n = n/2 · (2·3 + (n-1)·2) = n/2 · (6 + 2n - 2) = n/2 · (2n + 4).",
    "Simplify: S_n = n(n + 2).",
]

verify(arithmetic_sum, my_arith_formula, label="arithmetic series a=3, d=2", steps=steps)


✓ PASS — arithmetic series a=3, d=2
  your answer:   n*(n + 2)
  correct form:  n*(n + 2)
  method:        symbolic simplify



True

## Complex numbers

Verify modulus, argument, or algebraic manipulation of complex expressions.

In [10]:
# Example: (2 + 3i)(1 - i), then its modulus
z1 = (2 + 3*I) * (1 - I)
my_expanded = 5 + I
my_modulus = sqrt(26)

expand_steps = [
    "Expand with FOIL: (2+3i)(1-i) = 2(1) + 2(-i) + 3i(1) + 3i(-i).",
    "= 2 - 2i + 3i - 3i^2.",
    "Since i^2 = -1, -3i^2 = 3, so the real parts combine: 2 + 3 = 5, imaginary parts: -2i + 3i = i.",
    "Total: 5 + i.",
]
modulus_steps = [
    "Modulus is multiplicative: |z1 z2| = |z1| |z2|.",
    "|2+3i| = √(2² + 3²) = √13.  |1-i| = √(1² + 1²) = √2.",
    "So |(2+3i)(1-i)| = √13 · √2 = √26.",
]

verify(sp.expand(z1), my_expanded, label="(2+3i)(1-i) expanded", steps=expand_steps)
verify(Abs(z1), my_modulus, label="|(2+3i)(1-i)|", steps=modulus_steps)


✓ PASS — (2+3i)(1-i) expanded
  your answer:   5 + I
  correct form:  5 + I
  method:        symbolic simplify

✓ PASS — |(2+3i)(1-i)|
  your answer:   sqrt(26)
  correct form:  sqrt(26)
  method:        symbolic simplify



True

## Vectors

Check dot products, cross products, or magnitudes computed by hand.

In [11]:
u = Matrix([1, -2, 3])
v = Matrix([4, 0, -1])

my_dot = 1
my_cross = Matrix([2, 13, 8])
my_magnitude_u = sqrt(14)

dot_steps = [
    "u·v = u1v1 + u2v2 + u3v3 = (1)(4) + (-2)(0) + (3)(-1).",
    "= 4 + 0 - 3 = 1.",
]
verify(u.dot(v), my_dot, label="u · v", steps=dot_steps)

cross_steps = [
    "u×v = | i  j  k ; 1 -2  3 ; 4  0 -1 | (expand along the top row).",
    "i-component: (-2)(-1) - (3)(0) = 2.",
    "j-component: -[(1)(-1) - (3)(4)] = -(-1 - 12) = 13.",
    "k-component: (1)(0) - (-2)(4) = 0 + 8 = 8.",
    "So u×v = (2, 13, 8).",
]
derived_cross = u.cross(v)
verdict = list(derived_cross) == list(my_cross)
diagnosis = None
if not verdict:
    residual = sp.Matrix(my_cross) - derived_cross
    diagnosis = f"Your cross product differs from the correct one by {list(residual)} component-wise — recheck the sign pattern (+,-,+) across i, j, k."
_report(verdict, "u × v", list(my_cross), list(derived_cross), steps=cross_steps, diagnosis=diagnosis)

magnitude_steps = [
    "|u| = √(u1² + u2² + u3²) = √(1² + (-2)² + 3²) = √(1 + 4 + 9) = √14.",
]
verify(u.norm(), my_magnitude_u, label="|u|", steps=magnitude_steps)


✓ PASS — u · v
  your answer:   1
  correct form:  1
  method:        symbolic simplify

✓ PASS — u × v
  your answer:   [2, 13, 8]
  correct form:  [2, 13, 8]

✓ PASS — |u|
  your answer:   sqrt(14)
  correct form:  sqrt(14)
  method:        symbolic simplify



True

## Your own problem

Copy this cell for a new question — replace `expression` and `claimed_answer`, and (optionally) write out `steps` as you work through it so a mistake gets coached instead of just flagged. Without `steps`, a FAIL still shows the correct form and a best-effort diagnosis of the gap — just less specific than when you supply your own working.

In [12]:
expression = x**2 + 2*x + 1      # <- the thing you're deriving from (a derivative, an integral, an equation's LHS-RHS, ...)
claimed_answer = (x + 1)**2      # <- the answer you worked out by hand
steps = [
    # "Step 1: ...",
    # "Step 2: ...",
]

verify(expression, claimed_answer, label="describe the problem here", steps=steps or None)


✓ PASS — describe the problem here
  your answer:   (x + 1)**2
  correct form:  x**2 + 2*x + 1
  method:        symbolic simplify



True